# Ingauge 

In [ ]:
import os
import re
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from scipy.stats import entropy, spearmanr, kruskal, zscore
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    silhouette_score,
)

warnings.filterwarnings("ignore")

In [1]:
base = ***

input_dirs = [
    os.path.join(base, "longitudinal"),
    os.path.join(base, "participant_class_info"),
    os.path.join(base, "survey"),
    os.path.join(base, "class_wearable_data")
]

output_dir = ***
os.makedirs(output_dir, exist_ok=True)

column_inventory_output = os.path.join(output_dir, "column_inventory_by_file.csv")
missingness_output = os.path.join(output_dir, "missingness_by_file_and_column.csv")
temporal_audit_output = os.path.join(output_dir, "temporal_column_audit.csv")
all_columns_output = os.path.join(output_dir, "all_unique_columns.csv")

def clean_columns(cols):
    return (
        pd.Series(cols)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^\w_]", "", regex=True)
        .tolist()
    )

def find_csvs(folders):
    csvs = []
    for folder in folders:
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith(".csv"):
                    csvs.append(os.path.join(root, file))
    return csvs

time_keywords = [
    "time", "date", "timestamp", "datetime",
    "hour", "minute", "second", "day", "week"
]

def looks_temporal(col):
    c = col.lower()
    return any(k in c for k in time_keywords)

csv_files = find_csvs(input_dirs)

print(f"CSV files found: {len(csv_files)}")

column_inventory = []
missingness_rows = []
temporal_rows = []
all_columns = set()

for i, path in enumerate(csv_files, start=1):
    try:

        rel_path = os.path.relpath(path, base)
        source_folder = rel_path.split(os.sep)[0]
        source_file = os.path.basename(path)

        df = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip"
        )

        df.columns = clean_columns(df.columns)

        n_rows, n_cols = df.shape

    
        for col in df.columns:
            all_columns.add(col)

      
        column_inventory.append({
            "source_folder": source_folder,
            "source_file": source_file,
            "source_relative_path": rel_path,
            "n_rows": n_rows,
            "n_columns": n_cols,
            "columns": "; ".join(df.columns)
        })

      
        miss = df.isna().sum()
        miss_pct = df.isna().mean() * 100

        for col in df.columns:

            missingness_rows.append({
                "source_folder": source_folder,
                "source_file": source_file,
                "source_relative_path": rel_path,
                "column": col,
                "dtype": str(df[col].dtype),
                "n_rows": n_rows,
                "missing_count": int(miss[col]),
                "missing_percent": float(miss_pct[col]),
                "non_missing_count": int(n_rows - miss[col])
            })

        temporal_cols = [
            c for c in df.columns
            if looks_temporal(c)
        ]

        for col in temporal_cols:

            s = df[col].dropna()

            # Sample large columns
            if len(s) > 5000:
                s_sample = s.sample(
                    n=5000,
                    random_state=42
                )
            else:
                s_sample = s

            sample_values = (
                s_sample.astype(str)
                .drop_duplicates()
                .head(10)
                .tolist()
            )

            parsed = pd.to_datetime(
                s_sample.astype(str),
                errors="coerce"
            )

            if len(s_sample) > 0:
                parse_success = (
                    parsed.notna().mean() * 100
                )
            else:
                parse_success = np.nan

            temporal_rows.append({
                "source_folder": source_folder,
                "source_file": source_file,
                "source_relative_path": rel_path,
                "column": col,
                "dtype": str(df[col].dtype),
                "n_rows": n_rows,
                "non_missing_count": int(len(s)),
                "unique_values": int(s.nunique()),
                "datetime_parse_success_percent_sampled": parse_success,
                "sample_n_used_for_datetime_parse": int(len(s_sample)),
                "sample_values": " | ".join(sample_values)
            })

        del df
        gc.collect()

        if i % 100 == 0:
            print(f"Processed {i} files")

    except Exception as e:
        
        print(f"FAILED: {path}")
        print(e)

pd.DataFrame(column_inventory).to_csv(column_inventory_output, index=False)
pd.DataFrame(missingness_rows).to_csv(missingness_output, index=False)
pd.DataFrame(temporal_rows).to_csv(temporal_audit_output, index=False)

pd.DataFrame({
    "column": sorted(all_columns)
}).to_csv(all_columns_output, index=False)

print("\nDone.")
print(f"Column inventory saved to: {column_inventory_output}")
print(f"Missingness saved to: {missingness_output}")
print(f"Temporal audit saved to: {temporal_audit_output}")
print(f"All unique columns saved to: {all_columns_output}")

env_path = os.path.join(base, "longitudinal", "all_environment.csv")

env = pd.read_csv(env_path, low_memory=False)

env.columns = (
    env.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

env["time_parsed"] = pd.to_datetime(
    env["time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
).dt.time

env["time_seconds"] = pd.to_timedelta(env["time"].astype(str), errors="coerce").dt.total_seconds()
env["time_minutes"] = env["time_seconds"] / 60

env["study_day"] = pd.to_numeric(env["day"], errors="coerce")

env["elapsed_minutes"] = (
    env["study_day"] * 1440 +
    env["time_minutes"]
)

env["elapsed_hours"] = env["elapsed_minutes"] / 60

env["environment_datetime_index"] = pd.to_datetime(
    "2000-01-01"
) + pd.to_timedelta(env["elapsed_minutes"], unit="m")

env.to_csv(
    os.path.join(output_dir, "environment_5min_cleaned.csv"),
    index=False
)

print("Saved environment_5min_cleaned.csv")
print(env[["source_id", "source_file", "day", "time", "hour", "elapsed_minutes", "environment_datetime_index"]].head())


class_path = os.path.join(base, "participant_class_info", "class_table.csv")

classes = pd.read_csv(class_path, low_memory=False)

classes.columns = (
    classes.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

for col in ["start_time", "finish_time"]:
    classes[col + "_parsed"] = pd.to_datetime(
        classes[col].astype(str),
        errors="coerce"
    )

classes["class_len_minutes"] = pd.to_numeric(
    classes["class_len"],
    errors="coerce"
)

classes.to_csv(
    os.path.join(output_dir, "class_table_cleaned.csv"),
    index=False
)

print("Saved class_table_cleaned.csv")
print(classes.head())

student_survey_path = os.path.join(base, "survey", "student_survey.csv")

student_survey = pd.read_csv(student_survey_path, low_memory=False)

student_survey.columns = (
    student_survey.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

student_survey["survey_time_parsed"] = pd.to_datetime(
    student_survey["time"].astype(str),
    errors="coerce"
)

student_survey.to_csv(
    os.path.join(output_dir, "student_survey_cleaned.csv"),
    index=False
)

print("Saved student_survey_cleaned.csv")

teacher_survey_path = os.path.join(base, "survey", "teacher_survey.csv")

teacher_survey = pd.read_csv(teacher_survey_path, low_memory=False)

teacher_survey.columns = (
    teacher_survey.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

teacher_survey["survey_time_parsed"] = pd.to_datetime(
    teacher_survey["time"].astype(str),
    errors="coerce"
)

teacher_survey.to_csv(
    os.path.join(output_dir, "teacher_survey_cleaned.csv"),
    index=False
)

print("Saved teacher_survey_cleaned.csv")

wearable_time_audit_output = os.path.join(output_dir, "wearable_time_audit.csv")
wearable_time_samples_output = os.path.join(output_dir, "wearable_time_samples.csv")

def find_wearable_csvs(folder):
    files = []
    for root, _, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith(".csv"):
                files.append(os.path.join(root, f))
    return files

def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

def extract_ids(path):
    rel = os.path.relpath(path, wearable_base)
    parts = rel.split(os.sep)

    pid_folder = parts[0] if len(parts) > 0 else None
    class_or_session_folder = parts[1] if len(parts) > 1 else None
    file_name = parts[-1]

    signal = os.path.splitext(file_name)[0].lower()

    return rel, pid_folder, class_or_session_folder, signal

def classify_time_column(s):
    s_nonnull = s.dropna()

    if len(s_nonnull) == 0:
        return "empty"

    s_str = s_nonnull.astype(str).head(5000)

    numeric = pd.to_numeric(s_str, errors="coerce")
    numeric_success = numeric.notna().mean() * 100

    datetime_parsed = pd.to_datetime(s_str, errors="coerce")
    datetime_success = datetime_parsed.notna().mean() * 100

    clock_like = s_str.str.match(r"^\d{1,2}:\d{2}(:\d{2})?$").mean() * 100

    if numeric_success >= 95:
        return "numeric_elapsed_or_epoch"
    elif clock_like >= 95:
        return "clock_time"
    elif datetime_success >= 95:
        return "datetime"
    else:
        return "unclear_or_mixed"

files = find_wearable_csvs(wearable_base)

print(f"Wearable CSV files found: {len(files)}")

audit_rows = []
sample_rows = []

for i, path in enumerate(files, start=1):

    try:
        rel_path, pid_folder, class_session_folder, signal = extract_ids(path)

        df = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip"
        )

        df = clean_columns(df)

        if "time" not in df.columns:
            audit_rows.append({
                "source_relative_path": rel_path,
                "pid_folder": pid_folder,
                "class_or_session_folder": class_session_folder,
                "signal": signal,
                "has_time": False,
                "n_rows": df.shape[0],
                "n_columns": df.shape[1]
            })
            continue

        t = df["time"].dropna()
        t_num = pd.to_numeric(t, errors="coerce")

        valid_num = t_num.dropna()

        first_values = t.astype(str).head(10).tolist()
        last_values = t.astype(str).tail(10).tolist()

        if len(valid_num) >= 2:
            diffs = valid_num.diff().dropna()
            positive_diffs = diffs[diffs > 0]

            median_step = positive_diffs.median() if len(positive_diffs) > 0 else np.nan
            min_time = valid_num.min()
            max_time = valid_num.max()
            duration = max_time - min_time

            estimated_hz = 1 / median_step if pd.notna(median_step) and median_step > 0 else np.nan
        else:
            median_step = np.nan
            min_time = np.nan
            max_time = np.nan
            duration = np.nan
            estimated_hz = np.nan

        time_type = classify_time_column(t)

        audit_rows.append({
            "source_relative_path": rel_path,
            "pid_folder": pid_folder,
            "class_or_session_folder": class_session_folder,
            "signal": signal,
            "has_time": True,
            "n_rows": df.shape[0],
            "n_columns": df.shape[1],
            "time_dtype": str(df["time"].dtype),
            "time_type_guess": time_type,
            "time_non_missing": int(t.shape[0]),
            "time_unique": int(t.nunique()),
            "time_min_numeric": min_time,
            "time_max_numeric": max_time,
            "time_duration_numeric": duration,
            "median_positive_step": median_step,
            "estimated_hz_from_time": estimated_hz,
            "first_10_time_values": " | ".join(first_values),
            "last_10_time_values": " | ".join(last_values)
        })

        sample = df.head(20).copy()
        sample["source_relative_path"] = rel_path
        sample["pid_folder"] = pid_folder
        sample["class_or_session_folder"] = class_session_folder
        sample["signal"] = signal

        sample_rows.append(sample)

        del df
        gc.collect()

        if i % 100 == 0:
            print(f"Processed {i} wearable files")

    except Exception as e:
        print(f"FAILED: {path}")
        print(e)


audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(wearable_time_audit_output, index=False)

if len(sample_rows) > 0:
    samples_df = pd.concat(sample_rows, ignore_index=True, sort=False)
    samples_df.to_csv(wearable_time_samples_output, index=False)

print("\nDone.")
print(f"Wearable time audit saved to: {wearable_time_audit_output}")
print(f"Wearable time samples saved to: {wearable_time_samples_output}")

print("\nTime type counts:")
print(audit_df["time_type_guess"].value_counts(dropna=False))

print("\nEstimated Hz by signal:")
print(
    audit_df.groupby("signal")["estimated_hz_from_time"]
    .describe()
)

audit_path = ***

audit = pd.read_csv(audit_path)

cols = [
    "signal",
    "time_type_guess",
    "first_10_time_values",
    "last_10_time_values"
]

print(
    audit[cols]
    .drop_duplicates()
    .head(50)
    .to_string(index=False)
)

def find_wearable_csvs(folder):
    files = []
    for root, _, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith(".csv"):
                files.append(os.path.join(root, f))
    return files

def extract_ids(path):
    rel = os.path.relpath(path, wearable_base)
    parts = rel.split(os.sep)

    pid = parts[0] if len(parts) > 0 else None
    class_or_session = parts[1] if len(parts) > 1 else None
    signal = os.path.splitext(parts[-1])[0].lower()

    return rel, pid, class_or_session, signal

def time_to_seconds(series):
    td = pd.to_timedelta(series.astype(str), errors="coerce")
    return td.dt.total_seconds()

files = find_wearable_csvs(wearable_base)

rows = []

for i, path in enumerate(files, start=1):
    try:
        rel, pid, class_session, signal = extract_ids(path)

        df = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip"
        )

        df.columns = (
            df.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(" ", "_", regex=False)
            .str.replace("-", "_", regex=False)
            .str.replace(r"[^\w_]", "", regex=True)
        )

        if "time" not in df.columns:
            continue

        t_sec = time_to_seconds(df["time"])
        t_valid = t_sec.dropna()

        if len(t_valid) >= 2:
            diffs = t_valid.diff().dropna()
            positive_diffs = diffs[diffs > 0]

            median_step_sec = positive_diffs.median()
            estimated_hz = 1 / median_step_sec if median_step_sec > 0 else np.nan

            start_sec = t_valid.iloc[0]
            end_sec = t_valid.iloc[-1]
            duration_sec = end_sec - start_sec

            # Handle recordings that cross midnight
            if pd.notna(duration_sec) and duration_sec < 0:
                duration_sec += 24 * 3600

            start_clock = str(df["time"].dropna().iloc[0])
            end_clock = str(df["time"].dropna().iloc[-1])
        else:
            median_step_sec = np.nan
            estimated_hz = np.nan
            start_sec = np.nan
            end_sec = np.nan
            duration_sec = np.nan
            start_clock = None
            end_clock = None

        rows.append({
            "source_relative_path": rel,
            "pid": pid,
            "class_or_session": class_session,
            "signal": signal,
            "n_rows": df.shape[0],
            "start_clock": start_clock,
            "end_clock": end_clock,
            "start_seconds_since_midnight": start_sec,
            "end_seconds_since_midnight": end_sec,
            "duration_seconds": duration_sec,
            "duration_minutes": duration_sec / 60 if pd.notna(duration_sec) else np.nan,
            "median_step_seconds": median_step_sec,
            "estimated_hz": estimated_hz
        })

        del df
        gc.collect()

        if i % 200 == 0:
            print(f"Processed {i} files")

    except Exception as e:
        print(f"FAILED: {path}")
        print(e)

audit = pd.DataFrame(rows)
audit.to_csv(output_path, index=False)

print("Done.")
print(f"Saved: {output_path}")

print("\nEstimated Hz by signal:")
print(
    audit.groupby("signal")["estimated_hz"]
    .describe()
)

print("\nDuration minutes by signal:")
print(
    audit.groupby("signal")["duration_minutes"]
    .describe()
)

WINDOW_SECONDS = 60

def find_wearable_csvs(folder):
    files = []

    for root, _, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith(".csv"):
                files.append(os.path.join(root, f))

    return files


def extract_ids(path):

    rel = os.path.relpath(path, wearable_base)

    parts = rel.split(os.sep)

    pid = parts[0] if len(parts) > 0 else None
    session = parts[1] if len(parts) > 1 else None

    signal = os.path.splitext(parts[-1])[0].lower()

    return rel, pid, session, signal


def clean_columns(df):

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^\w_]", "", regex=True)
    )

    return df


def convert_time_to_seconds(series):

    td = pd.to_timedelta(series.astype(str), errors="coerce")

    return td.dt.total_seconds()


def fix_midnight_rollover(seconds_series):

    vals = seconds_series.copy()

    if len(vals) < 2:
        return vals

    adjusted = vals.copy()

    for i in range(1, len(adjusted)):

        if pd.isna(adjusted.iloc[i - 1]) or pd.isna(adjusted.iloc[i]):
            continue

        if adjusted.iloc[i] < adjusted.iloc[i - 1]:
            adjusted.iloc[i:] += 24 * 3600

    return adjusted


def compute_basic_stats(x, prefix):

    out = {}

    x = pd.Series(x).dropna()

    if len(x) == 0:
        return out

    out[f"{prefix}_mean"] = x.mean()
    out[f"{prefix}_std"] = x.std()
    out[f"{prefix}_min"] = x.min()
    out[f"{prefix}_max"] = x.max()
    out[f"{prefix}_median"] = x.median()

    return out


def compute_rmssd(ibi_values):

    ibi = pd.Series(ibi_values).dropna()

    if len(ibi) < 3:
        return np.nan

    diffs = np.diff(ibi)

    return np.sqrt(np.mean(diffs ** 2))


all_feature_rows = []

files = find_wearable_csvs(wearable_base)

print(f"Found {len(files)} wearable files")

for idx, path in enumerate(files, start=1):

    try:

        rel, pid, session, signal = extract_ids(path)

        df = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip"
        )

        df = clean_columns(df)

        if "time" not in df.columns:
            continue



        df["time_seconds"] = convert_time_to_seconds(df["time"])

        df = df.dropna(subset=["time_seconds"])

        if len(df) == 0:
            continue

        df["time_seconds"] = fix_midnight_rollover(
            df["time_seconds"]
        )

        start_time = df["time_seconds"].min()

        df["window_id"] = (
            (df["time_seconds"] - start_time)
            // WINDOW_SECONDS
        ).astype(int)


        if signal == "acc":

            if all(c in df.columns for c in ["x", "y", "z"]):

                df["acc_mag"] = np.sqrt(
                    df["x"] ** 2 +
                    df["y"] ** 2 +
                    df["z"] ** 2
                )

      

        grouped = df.groupby("window_id")

        for window_id, g in grouped:

            row = {
                "pid": pid,
                "session": session,
                "signal": signal,
                "window_id": window_id,
                "window_start_sec": g["time_seconds"].min(),
                "window_end_sec": g["time_seconds"].max(),
                "n_samples": len(g)
            }

         

            if signal == "hr":

                if "hr" in g.columns:
                    row.update(
                        compute_basic_stats(
                            g["hr"],
                            "hr"
                        )
                    )

            elif signal == "eda":

                if "eda" in g.columns:

                    row.update(
                        compute_basic_stats(
                            g["eda"],
                            "eda"
                        )
                    )

                    if len(g) > 1:
                        row["eda_slope"] = (
                            g["eda"].iloc[-1] -
                            g["eda"].iloc[0]
                        )

            elif signal == "temp":

                if "temp" in g.columns:

                    row.update(
                        compute_basic_stats(
                            g["temp"],
                            "temp"
                        )
                    )

                    if len(g) > 1:
                        row["temp_slope"] = (
                            g["temp"].iloc[-1] -
                            g["temp"].iloc[0]
                        )

            elif signal == "bvp":

                if "bvp" in g.columns:

                    row.update(
                        compute_basic_stats(
                            g["bvp"],
                            "bvp"
                        )
                    )

                    row["bvp_energy"] = np.sum(
                        g["bvp"] ** 2
                    )

            elif signal == "ibi":

                if "ibi" in g.columns:

                    row.update(
                        compute_basic_stats(
                            g["ibi"],
                            "ibi"
                        )
                    )

                    row["ibi_rmssd"] = compute_rmssd(
                        g["ibi"]
                    )

            elif signal == "acc":

                if "acc_mag" in g.columns:

                    row.update(
                        compute_basic_stats(
                            g["acc_mag"],
                            "acc_mag"
                        )
                    )

                    row["acc_activity_count"] = (
                        g["acc_mag"].diff().abs().sum()
                    )

            all_feature_rows.append(row)

        del df
        gc.collect()

        if idx % 200 == 0:
            print(f"Processed {idx} / {len(files)}")

    except Exception as e:

        print(f"\nFAILED: {path}")
        print(e)


features = pd.DataFrame(all_feature_rows)

features.to_csv(output_path, index=False)

print("\nDONE")
print(f"Saved features to:\n{output_path}")

print("\nFeature table shape:")
print(features.shape)

print("\nSignals:")
print(features['signal'].value_counts())

print("\nColumns:")
print(features.columns.tolist())

index_cols = [
    "pid",
    "session",
    "window_id"
]

non_feature_cols = [
    "signal",
    "window_start_sec",
    "window_end_sec",
    "n_samples"
]

feature_cols = [
    c for c in df.columns
    if c not in index_cols + non_feature_cols
]


timing = (
    df.groupby(index_cols)
    .agg({
        "window_start_sec": "min",
        "window_end_sec": "max"
    })
    .reset_index()
)

signal_tables = []

signals = sorted(df["signal"].unique())

for signal in signals:

    sub = df[df["signal"] == signal].copy()

    keep_cols = index_cols + feature_cols

    keep_cols = [
        c for c in keep_cols
        if c in sub.columns
    ]

    sub = sub[keep_cols]

    # Remove fully empty columns
    sub = sub.dropna(axis=1, how="all")

    signal_tables.append(sub)


merged = timing.copy()

for s in signal_tables:

    merged = merged.merge(
        s,
        on=index_cols,
        how="outer"
    )


merged = merged.sort_values(
    ["pid", "session", "window_id"]
).reset_index(drop=True)


merged.to_csv(output_path, index=False)

print("DONE")
print(f"Saved wide table to:\n{output_path}")

print("\nShape:")
print(merged.shape)

print("\nColumns:")
print(merged.columns.tolist())

wearable_path = ***
class_path = ***

output_path = o***

wear = pd.read_csv(wearable_path)
classes = pd.read_csv(class_path)

classes.columns = (
    classes.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

wear["pid"] = pd.to_numeric(wear["pid"], errors="coerce")
wear["session"] = pd.to_numeric(wear["session"], errors="coerce")

classes["class_id"] = pd.to_numeric(classes["class_id"], errors="coerce")

merged = wear.merge(
    classes,
    left_on="session",
    right_on="class_id",
    how="left"
)

merged["window_start_clock"] = (
    pd.to_datetime("2000-01-01") +
    pd.to_timedelta(merged["window_start_sec"], unit="s")
)

merged["window_end_clock"] = (
    pd.to_datetime("2000-01-01") +
    pd.to_timedelta(merged["window_end_sec"], unit="s")
)

merged.to_csv(output_path, index=False)

print("DONE")
print(f"Saved: {output_path}")
print("Shape:", merged.shape)
print("Class metadata match rate:")
print(merged["class_id"].notna().mean() * 100)

wearable_class_path = ***
env_path = ***

wear = pd.read_csv(wearable_class_path)
env = pd.read_csv(env_path, low_memory=False)

wear.columns = (
    wear.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

env.columns = (
    env.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)


env["env_time_sec"] = pd.to_timedelta(
    env["time"].astype(str),
    errors="coerce"
).dt.total_seconds()

env["env_time_min"] = env["env_time_sec"] / 60

env["env_room_source"] = (
    env["source_file"]
    .astype(str)
    .str.replace(".csv", "", regex=False)
)

wear["window_mid_sec"] = (
    wear["window_start_sec"] + wear["window_end_sec"]
) / 2

wear["window_mid_min"] = wear["window_mid_sec"] / 60

wear["env_time_min"] = (
    (wear["window_mid_min"] / 5)
    .round()
    * 5
)

wear["env_time_min"] = wear["env_time_min"] % (24 * 60)

env["env_time_min"] = env["env_time_min"].round(6)
wear["env_time_min"] = wear["env_time_min"].round(6)

env_keep = [
    "source_id",
    "source_file",
    "env_room_source",
    "day",
    "time",
    "env_time_min",
    "occupied",
    "schoolday",
    "hour",
    "lessonnumber",
    "lessonpct",
    "indoortemperature",
    "indoorhumidity",
    "indoorco2",
    "indoornoise",
    "outdoortemperature",
    "outdoorhumidity",
    "outdoordewpoint",
    "outdoorwinddirection",
    "outdoorwindspeed",
    "outdoorgustspeed",
    "precipitation",
    "uvlevel",
    "solarradiation",
    "coolingstate",
    "heatingstate",
    "usabilitymask"
]

env_keep = [c for c in env_keep if c in env.columns]

env_small = env[env_keep].copy()

env_small = env_small.rename(columns={
    "source_id": "env_source_id",
    "source_file": "env_source_file",
    "day": "env_day",
    "time": "env_time",
    "hour": "env_hour"
})

merged = wear.merge(
    env_small,
    left_on=["room", "env_time_min"],
    right_on=["env_room_source", "env_time_min"],
    how="left"
)

merged.to_csv(output_path, index=False)

print("DONE")
print(f"Saved: {output_path}")
print("Shape:", merged.shape)

print("\nEnvironment match rate:")
print(merged["env_source_file"].notna().mean() * 100)

print("\nMatched env files:")
print(merged["env_source_file"].value_counts(dropna=False).head(20))

print("\nColumns:")
print(merged.columns.tolist())

wear_path = os.path.join(output_dir, "wearable_windows_with_class_metadata.csv")
env_path = os.path.join(base, "longitudinal", "all_environment.csv")

output_path = os.path.join(output_dir, "wearable_windows_with_environment_time_only.csv")

wear = pd.read_csv(wear_path)
env = pd.read_csv(env_path, low_memory=False)

wear.columns = wear.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
env.columns = env.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")

wear["window_mid_sec"] = (wear["window_start_sec"] + wear["window_end_sec"]) / 2
wear["env_time_min"] = ((wear["window_mid_sec"] / 60) / 5).round() * 5
wear["env_time_min"] = wear["env_time_min"] % 1440


env["env_time_min"] = pd.to_timedelta(env["time"].astype(str), errors="coerce").dt.total_seconds() / 60
env["env_time_min"] = env["env_time_min"].round(6)

wear["study_day_est"] = ((wear["week"] - 1) * 7) + wear["weekday"]

env["study_day_est"] = env["day"]

env_agg = (
    env.groupby(["study_day_est", "env_time_min"], as_index=False)
    .agg({
        "occupied": "mean",
        "schoolday": "mean",
        "hour": "mean",
        "lessonnumber": "mean",
        "lessonpct": "mean",
        "indoortemperature": "mean",
        "indoorhumidity": "mean",
        "indoorco2": "mean",
        "indoornoise": "mean",
        "outdoortemperature": "mean",
        "outdoorhumidity": "mean",
        "outdoordewpoint": "mean",
        "outdoorwinddirection": "mean",
        "outdoorwindspeed": "mean",
        "outdoorgustspeed": "mean",
        "precipitation": "mean",
        "uvlevel": "mean",
        "solarradiation": "mean",
        "coolingstate": "mean",
        "heatingstate": "mean",
        "usabilitymask": "mean"
    })
)

env_agg = env_agg.rename(columns={
    "occupied": "env_occupied_mean",
    "schoolday": "env_schoolday_mean",
    "hour": "env_hour_mean",
    "lessonnumber": "env_lessonnumber_mean",
    "lessonpct": "env_lessonpct_mean",
    "indoortemperature": "env_indoortemperature_mean",
    "indoorhumidity": "env_indoorhumidity_mean",
    "indoorco2": "env_indoorco2_mean",
    "indoornoise": "env_indoornoise_mean",
    "outdoortemperature": "env_outdoortemperature_mean",
    "outdoorhumidity": "env_outdoorhumidity_mean",
    "outdoordewpoint": "env_outdoordewpoint_mean",
    "outdoorwinddirection": "env_outdoorwinddirection_mean",
    "outdoorwindspeed": "env_outdoorwindspeed_mean",
    "outdoorgustspeed": "env_outdoorgustspeed_mean",
    "precipitation": "env_precipitation_mean",
    "uvlevel": "env_uvlevel_mean",
    "solarradiation": "env_solarradiation_mean",
    "coolingstate": "env_coolingstate_mean",
    "heatingstate": "env_heatingstate_mean",
    "usabilitymask": "env_usabilitymask_mean"
})

merged = wear.merge(
    env_agg,
    on=["study_day_est", "env_time_min"],
    how="left"
)

merged.to_csv(output_path, index=False)

print("DONE")
print(f"Saved: {output_path}")
print("Shape:", merged.shape)

print("\nEnvironment match rate:")
print(merged["env_indoortemperature_mean"].notna().mean() * 100)

wear_path = os.path.join(output_dir, "wearable_windows_with_environment_time_only.csv")
survey_path = os.path.join(base, "survey", "student_survey.csv")
output_path = os.path.join(output_dir, "wearable_environment_survey_merged.csv")

wear = pd.read_csv(wear_path)
survey = pd.read_csv(survey_path)

wear.columns = wear.columns.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
survey.columns = survey.columns.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)

for col in ["pid", "week", "weekday"]:
    wear[col] = pd.to_numeric(wear[col], errors="coerce")
    survey[col] = pd.to_numeric(survey[col], errors="coerce")

survey["survey_time_sec"] = pd.to_timedelta(
    survey["time"].astype(str),
    errors="coerce"
).dt.total_seconds()

wear["window_mid_sec"] = (
    wear["window_start_sec"] +
    wear["window_end_sec"]
) / 2

merged_parts = []
group_cols = ["pid", "week", "weekday"]

for key, wear_g in wear.groupby(group_cols, dropna=False):

    pid, week, weekday = key

    survey_g = survey[
        (survey["pid"] == pid) &
        (survey["week"] == week) &
        (survey["weekday"] == weekday)
    ].copy()

    wear_g = wear_g.sort_values("window_mid_sec").copy()
    survey_g = survey_g.sort_values("survey_time_sec").copy()

    if survey_g.empty:
        for col in survey.columns:
            if col not in wear_g.columns:
                wear_g[col] = np.nan

        merged_parts.append(wear_g)
        continue

    merged_g = pd.merge_asof(
        wear_g,
        survey_g,
        left_on="window_mid_sec",
        right_on="survey_time_sec",
        direction="nearest",
        tolerance=15 * 60,
        suffixes=("", "_survey")
    )

    merged_parts.append(merged_g)

merged = pd.concat(
    merged_parts,
    ignore_index=True,
    sort=False
)

merged["survey_time_diff_min"] = (
    (merged["window_mid_sec"] - merged["survey_time_sec"]).abs() / 60
)

survey_cols = [
    "engage_1",
    "engage_2",
    "engage_3",
    "engage_4",
    "engage_5",
    "arousal",
    "valence",
    "confidence_level",
    "thermal_sensation",
    "thermal_preference"
]

survey_cols = [c for c in survey_cols if c in merged.columns]

merged["has_survey_match"] = (
    merged[survey_cols]
    .notna()
    .any(axis=1)
)

merged.to_csv(output_path, index=False)

print("DONE")
print(f"Saved: {output_path}")

print("\nShape:")
print(merged.shape)

print("\nSurvey match rate:")
print(merged["has_survey_match"].mean() * 100)

print("\nSurvey time difference:")
print(
    merged.loc[
        merged["has_survey_match"],
        "survey_time_diff_min"
    ].describe()
)

engage_cols = ["engage_1", "engage_2", "engage_3", "engage_4", "engage_5"]

likert_map = {
    "Strongly disagree": 1,
    "Somewhat disagree": 2,
    "Neither agree nor disagree": 3,
    "Somewhat agree": 4,
    "Strongly agree": 5
}

for col in engage_cols:
    df[col + "_num"] = (
        df[col]
        .astype(str)
        .str.strip()
        .map(likert_map)
    )

engage_num_cols = [c + "_num" for c in engage_cols]

df["engagement_mean"] = df[engage_num_cols].mean(axis=1)

df["engagement_high"] = (
    df["engagement_mean"] >= df["engagement_mean"].median()
).astype(int)

df.to_csv(output_path, index=False)

print("DONE")
print(f"Saved: {output_path}")
print("Shape:", df.shape)

print("\nEngagement numeric value counts:")
for col in engage_num_cols:
    print("\n", col)
    print(df[col].value_counts(dropna=False).sort_index())

print("\nTarget summaries:")
print(
    df[
        engage_num_cols +
        ["engagement_mean", "engagement_high", "thermal_sensation", "thermal_preference"]
    ].describe(include="all")
)

print("\nHigh engagement counts:")
print(df["engagement_high"].value_counts())

target_col = "engagement_high"

drop_cols = [
    # targets
    "engagement_high",
    "engagement_mean",

    # raw survey responses
    "engage_1",
    "engage_2",
    "engage_3",
    "engage_4",
    "engage_5",

    # numeric versions
    "engage_1_num",
    "engage_2_num",
    "engage_3_num",
    "engage_4_num",
    "engage_5_num",

    # unavailable targets
    "arousal",
    "valence",
    "confidence_level",

    # metadata leakage
    "window_id",
    "window_start_clock",
    "window_end_clock",
    "survey_time_sec",

    # identifiers
    "pid",
    "session"
]

drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target_col]


missing_pct = X.isnull().mean()

keep_cols = missing_pct[missing_pct < 0.50].index.tolist()

X = X[keep_cols]

print("\nRemaining columns:", len(X.columns))


numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced"
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

groups = df["pid"]

cv = GroupKFold(n_splits=5)

results = []

for model_name, model in models.items():

    print("\n==============================")
    print(model_name)
    print("==============================")

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    preds_proba = cross_val_predict(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    preds = (preds_proba >= 0.5).astype(int)

    roc = roc_auc_score(y, preds_proba)
    pr = average_precision_score(y, preds_proba)

    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)

    precision = precision_score(y, preds)
    recall = recall_score(y, preds)

    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()

    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC : {pr:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

    results.append({
        "model": model_name,
        "roc_auc": roc,
        "pr_auc": pr,
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn
    })

results_df = pd.DataFrame(results)

results_path = os.path.join(
    output_dir,
    "engagement_model_results.csv"
)

results_df.to_csv(results_path, index=False)

print("\nDONE")
print(results_df)

print("\nSaved:")
print(results_path)

group_cols = [
    "pid",
    "week",
    "weekday",
    "survey_time_sec"
]

print("\nUnique survey events:")
print(df[group_cols].drop_duplicates().shape[0])

print("\nWindows per survey event:")
counts = (
    df.groupby(group_cols)
    .size()
)

print(counts.describe())

print("\nTop duplicated survey events:")
print(counts.sort_values(ascending=False).head(20))

survey_keys = [
    "pid",
    "week",
    "weekday",
    "survey_time_sec"
]

exclude_cols = set([
    "pid",
    "session",
    "week",
    "weekday",
    "survey_time_sec",
    "window_id",
    "window_start_clock",
    "window_end_clock",

    # raw survey text
    "engage_1",
    "engage_2",
    "engage_3",
    "engage_4",
    "engage_5",

    # targets
    "engagement_high",
    "engagement_mean",
])

numeric_cols = []

for col in df.columns:

    if col in exclude_cols:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)

print("\nNumeric feature count:", len(numeric_cols))

agg_dict = {}

for col in numeric_cols:

    agg_dict[col] = [
        "mean",
        "std",
        "min",
        "max"
    ]

event_df = (
    df
    .groupby(survey_keys)
    .agg(agg_dict)
)

# flatten column names
event_df.columns = [
    f"{c[0]}_{c[1]}"
    for c in event_df.columns
]

event_df = event_df.reset_index()

target_cols = [
    "engagement_mean",
    "engagement_high",
    "thermal_sensation",
    "thermal_preference",

    "engage_1_num",
    "engage_2_num",
    "engage_3_num",
    "engage_4_num",
    "engage_5_num"
]

target_df = (
    df[survey_keys + target_cols]
    .drop_duplicates(subset=survey_keys)
)

event_df = event_df.merge(
    target_df,
    on=survey_keys,
    how="left"
)

event_df.to_csv(output_path, index=False)

print("\nDONE")
print("Saved:", output_path)

print("\nShape:")
print(event_df.shape)

print("\nUnique survey events:")
print(
    event_df[survey_keys]
    .drop_duplicates()
    .shape[0]
)

print("\nTarget distribution:")
print(
    event_df["engagement_high"]
    .value_counts()
)

print("\nColumns:")
print(event_df.columns.tolist()[:20])
print("...")

exclude_cols = [
    "pid",
    "week",
    "weekday",
    "survey_time_sec",

    "engagement_mean",
    "engagement_high",
    "thermal_sensation",
    "thermal_preference",

    "engage_1_num",
    "engage_2_num",
    "engage_3_num",
    "engage_4_num",
    "engage_5_num"
]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

X = df[feature_cols]

# keep numeric only
X = X.select_dtypes(include=[np.number])

print("Feature matrix:", X.shape)

imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("\nExplained variance:")
print(pca.explained_variance_ratio_)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "engagement_high": df["engagement_high"],
    "thermal_sensation": df["thermal_sensation"],
    "pid": df["pid"]
})

pca_path = os.path.join(
    output_dir,
    "survey_event_pca.csv"
)

pca_df.to_csv(pca_path, index=False)


plt.figure(figsize=(8,6))

scatter = plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["engagement_high"],
    s=120
)

for i, row in pca_df.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        str(row["pid"]),
        fontsize=9
    )

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.title("Wearable Physiology PCA")

plt.colorbar(label="High Engagement")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "survey_event_pca.png"
)

plt.savefig(plot_path, dpi=300)

print("\nSaved PCA:")
print(pca_path)

print("\nSaved figure:")
print(plot_path)

exclude_cols = [
    "pid", "week", "weekday", "survey_time_sec",
    "engagement_mean", "engagement_high",
    "thermal_sensation", "thermal_preference",
    "engage_1_num", "engage_2_num", "engage_3_num",
    "engage_4_num", "engage_5_num"
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].select_dtypes(include=[np.number])

# Remove columns that are completely missing
X = X.dropna(axis=1, how="all")

# Remove zero-variance columns
X = X.loc[:, X.nunique(dropna=True) > 1]

print("Final feature matrix:", X.shape)

imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

pca = PCA(n_components=2)
pca.fit(X_scaled)

loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1_loading", "PC2_loading"],
    index=X.columns
)

loadings["PC1_abs"] = loadings["PC1_loading"].abs()
loadings["PC2_abs"] = loadings["PC2_loading"].abs()

print("\nExplained variance:")
print(pca.explained_variance_ratio_)

print("\nTOP PC1 FEATURES")
print(loadings.sort_values("PC1_abs", ascending=False)[["PC1_loading"]].head(20))

print("\nTOP PC2 FEATURES")
print(loadings.sort_values("PC2_abs", ascending=False)[["PC2_loading"]].head(20))

loadings_path = os.path.join(output_dir, "pca_feature_loadings.csv")
loadings.to_csv(loadings_path)

print("\nSaved:")
print(loadings_path)

candidate_features = [

  
    "acc_mag_mean_mean",
    "acc_mag_std_mean",
    "acc_activity_count_mean",

 
    "hr_mean_mean",
    "hr_min_mean",
    "hr_max_mean",

    
    "ibi_mean_mean",
    "ibi_std_mean",
    "ibi_rmssd_mean",

    
    "eda_mean_mean",
    "eda_std_mean",
    "eda_slope_mean",

   
    "temp_mean_mean",
    "temp_std_mean",
    "temp_slope_mean",

   
    "env_indoorco2_mean_mean",
    "env_indoorhumidity_mean_mean",
    "env_indoortemperature_mean_mean",
    "env_indoornoise_mean_mean",

    "env_lessonnumber_mean_mean",
    "env_lessonpct_mean_mean"
]

candidate_features = [
    c for c in candidate_features
    if c in df.columns
]

print("\nFeatures used:")
print(candidate_features)


targets = [
    "engagement_mean",
    "thermal_sensation"
]

results = []

for target in targets:

    for feature in candidate_features:

        sub = df[[target, feature]].dropna()

        if len(sub) < 5:
            continue

        rho, p = spearmanr(
            sub[target],
            sub[feature]
        )

        results.append({
            "target": target,
            "feature": feature,
            "rho": rho,
            "p": p
        })

results_df = pd.DataFrame(results)

results_df["fdr"] = multipletests(
    results_df["p"],
    method="fdr_bh"
)[1]

results_df["abs_rho"] = results_df["rho"].abs()

results_df = results_df.sort_values(
    ["target", "abs_rho"],
    ascending=[True, False]
)


results_path = os.path.join(
    output_dir,
    "physiology_environment_correlations.csv"
)

results_df.to_csv(results_path, index=False)

print("\nTOP CORRELATIONS")
print(results_df.head(30))

print("\nSaved:")
print(results_path)


heatmap_df = (
    results_df
    .pivot(
        index="feature",
        columns="target",
        values="rho"
    )
)

plt.figure(figsize=(8,10))

im = plt.imshow(
    heatmap_df.values,
    aspect="auto"
)

plt.colorbar(im, label="Spearman rho")

plt.xticks(
    range(len(heatmap_df.columns)),
    heatmap_df.columns,
    rotation=45
)

plt.yticks(
    range(len(heatmap_df.index)),
    heatmap_df.index
)

plt.title("Physiology-Environment Correlations")

plt.tight_layout()

heatmap_path = os.path.join(
    output_dir,
    "physiology_environment_heatmap.png"
)

plt.savefig(
    heatmap_path,
    dpi=300
)

print("\nSaved heatmap:")
print(heatmap_path)

top_positive = (
    df.sort_values("rho", ascending=False)
    .head(10)
)

top_negative = (
    df.sort_values("rho", ascending=True)
    .head(10)
)

plot_df = pd.concat([
    top_positive,
    top_negative
]).drop_duplicates()

plot_df["label"] = (
    plot_df["target"] +
    " | " +
    plot_df["feature"]
)

plot_df = plot_df.sort_values("rho")

table_path = os.path.join(
    output_dir,
    "top_positive_negative_correlations.csv"
)

plot_df.to_csv(table_path, index=False)

print("\nSaved table:")
print(table_path)

heatmap_matrix = plot_df[["rho"]].values


fig_height = max(6, len(plot_df) * 0.45)

plt.figure(figsize=(8, fig_height))

im = plt.imshow(
    heatmap_matrix,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Spearman rho")

plt.yticks(
    range(len(plot_df)),
    plot_df["label"]
)

plt.xticks([0], ["Correlation"])

plt.title(
    "Top Positive and Negative Physiology Correlations"
)

for i, rho in enumerate(plot_df["rho"]):

    plt.text(
        0,
        i,
        f"{rho:.2f}",
        ha="center",
        va="center",
        fontsize=9
    )

plt.tight_layout()

heatmap_path = os.path.join(
    output_dir,
    "top_positive_negative_heatmap.png"
)

plt.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches="tight"
)

print("\nSaved heatmap:")
print(heatmap_path)

df["window_mid_sec"] = (
    df["window_start_sec"] +
    df["window_end_sec"]
) / 2

df["window_mid_hour"] = df["window_mid_sec"] / 3600

df["study_day_est"] = (
    (pd.to_numeric(df["week"], errors="coerce") - 1) * 7 +
    pd.to_numeric(df["weekday"], errors="coerce")
)

df["absolute_time_hours"] = (
    df["study_day_est"] * 24 +
    df["window_mid_hour"]
)

core_vars = [
    "hr_mean",
    "ibi_mean",
    "ibi_rmssd",
    "eda_mean",
    "eda_slope",
    "temp_mean",
    "acc_mag_mean",
    "acc_activity_count",
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean",
    "env_lessonnumber_mean",
    "env_lessonpct_mean"
]

core_vars = [c for c in core_vars if c in df.columns]

keep_cols = [
    "pid",
    "session",
    "week",
    "weekday",
    "subject",
    "room",
    "class_id",
    "window_id",
    "window_mid_sec",
    "window_mid_hour",
    "absolute_time_hours"
] + core_vars

traj = df[keep_cols].copy()

physio_vars = [
    "hr_mean",
    "ibi_mean",
    "ibi_rmssd",
    "eda_mean",
    "eda_slope",
    "temp_mean",
    "acc_mag_mean",
    "acc_activity_count"
]

physio_vars = [c for c in physio_vars if c in traj.columns]

for col in physio_vars:
    traj[col + "_z"] = (
        traj[col] -
        traj.groupby("pid")[col].transform("mean")
    ) / traj.groupby("pid")[col].transform("std")

traj.to_csv(trajectory_output, index=False)

print("Saved trajectory dataset:")
print(trajectory_output)
print("Shape:", traj.shape)

def plot_population_trajectory(data, x_col, y_col, title, output_name):
    plot_df = data[[x_col, y_col]].dropna().copy()

    if plot_df.empty:
        print(f"Skipping {y_col}: no data")
        return

    # Bin x-axis for smoother trajectory
    plot_df["x_bin"] = pd.cut(
        plot_df[x_col],
        bins=20
    )

    summary = (
        plot_df
        .groupby("x_bin", observed=True)
        .agg(
            x_mean=(x_col, "mean"),
            y_mean=(y_col, "mean"),
            y_sem=(y_col, lambda x: x.std() / np.sqrt(len(x)))
        )
        .reset_index()
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        summary["x_mean"],
        summary["y_mean"],
        marker="o"
    )

    plt.fill_between(
        summary["x_mean"],
        summary["y_mean"] - summary["y_sem"],
        summary["y_mean"] + summary["y_sem"],
        alpha=0.2
    )

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title)

    plt.tight_layout()

    out_path = os.path.join(output_dir, output_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)

if "env_lessonpct_mean" in traj.columns:

    for var in physio_vars:

        z_col = var + "_z"

        if z_col in traj.columns:
            plot_population_trajectory(
                traj,
                x_col="env_lessonpct_mean",
                y_col=z_col,
                title=f"{var} Trajectory Across Lesson Progress",
                output_name=f"trajectory_lessonpct_{var}.png"
            )

for var in physio_vars:

    z_col = var + "_z"

    if z_col in traj.columns:
        plot_population_trajectory(
            traj,
            x_col="window_mid_hour",
            y_col=z_col,
            title=f"{var} Trajectory Across Time of Day",
            output_name=f"trajectory_timeofday_{var}.png"
        )

def plot_by_subject(data, x_col, y_col, output_name):
    plot_df = data[[x_col, y_col, "subject"]].dropna().copy()

    if plot_df.empty:
        print(f"Skipping {y_col}: no data")
        return

    plt.figure(figsize=(9, 6))

    for subject, g in plot_df.groupby("subject"):
        g = g.sort_values(x_col)

        # Bin within subject
        g["x_bin"] = pd.cut(g[x_col], bins=15)

        summary = (
            g.groupby("x_bin", observed=True)
            .agg(
                x_mean=(x_col, "mean"),
                y_mean=(y_col, "mean")
            )
            .reset_index()
        )

        plt.plot(
            summary["x_mean"],
            summary["y_mean"],
            marker="o",
            label=str(subject)
        )

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"{y_col} by Subject")
    plt.legend(fontsize=8)

    plt.tight_layout()

    out_path = os.path.join(output_dir, output_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)

if "env_lessonpct_mean" in traj.columns and "subject" in traj.columns:

    for var in ["hr_mean", "ibi_rmssd", "eda_mean", "temp_mean", "acc_activity_count"]:

        z_col = var + "_z"

        if z_col in traj.columns:
            plot_by_subject(
                traj,
                x_col="env_lessonpct_mean",
                y_col=z_col,
                output_name=f"trajectory_by_subject_{var}.png"
            )

print("\nDONE")

latent_features = [
    # autonomic
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",

    # sympathetic / arousal
    "eda_mean_z",
    "eda_slope_z",

    # thermal
    "temp_mean_z",

    # activity
    "acc_mag_mean_z",
    "acc_activity_count_z",

    # environment
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean",
    "env_lessonnumber_mean",
    "env_lessonpct_mean"
]

latent_features = [
    c for c in latent_features
    if c in df.columns
]

print("Latent features:")
print(latent_features)

X = df[latent_features].copy()

# Remove empty / constant columns
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

print("Final latent feature matrix:", X.shape)

imputer = SimpleImputer(strategy="median")
X_imp = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("\nExplained variance:")
print(pca.explained_variance_ratio_)

df["latent_pc1"] = X_pca[:, 0]
df["latent_pc2"] = X_pca[:, 1]
df["latent_pc3"] = X_pca[:, 2]

df.to_csv(latent_output, index=False)

print("\nSaved latent trajectory dataset:")
print(latent_output)

plt.figure(figsize=(8, 6))

sc = plt.scatter(
    df["latent_pc1"],
    df["latent_pc2"],
    c=df["window_mid_hour"],
    s=10,
    alpha=0.7
)

plt.colorbar(sc, label="Time of day / window_mid_hour")
plt.xlabel("Latent PC1")
plt.ylabel("Latent PC2")
plt.title("Latent Physiologic State Space Colored by Time")

plt.tight_layout()

out_path = os.path.join(
    output_dir,
    "latent_state_pca_by_time.png"
)

plt.savefig(out_path, dpi=300)
plt.close()

print("Saved:", out_path)

if "env_lessonpct_mean" in df.columns:

    plt.figure(figsize=(8, 6))

    sc = plt.scatter(
        df["latent_pc1"],
        df["latent_pc2"],
        c=df["env_lessonpct_mean"],
        s=10,
        alpha=0.7
    )

    plt.colorbar(sc, label="Lesson progress")
    plt.xlabel("Latent PC1")
    plt.ylabel("Latent PC2")
    plt.title("Latent Physiologic State Space Colored by Lesson Progress")

    plt.tight_layout()

    out_path = os.path.join(
        output_dir,
        "latent_state_pca_by_lessonpct.png"
    )

    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)

if "subject" in df.columns:

    subjects = sorted(df["subject"].dropna().unique())

    plt.figure(figsize=(9, 7))

    for subject in subjects:
        g = df[df["subject"] == subject]

        plt.scatter(
            g["latent_pc1"],
            g["latent_pc2"],
            s=12,
            alpha=0.6,
            label=str(subject)
        )

    plt.xlabel("Latent PC1")
    plt.ylabel("Latent PC2")
    plt.title("Latent Physiologic State Space by Subject")
    plt.legend(fontsize=8)

    plt.tight_layout()

    out_path = os.path.join(
        output_dir,
        "latent_state_pca_by_subject.png"
    )

    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)

plot_df = df.dropna(
    subset=["latent_pc1", "latent_pc2", "pid", "absolute_time_hours"]
).copy()

plot_df = plot_df.sort_values(
    ["pid", "absolute_time_hours"]
)

plt.figure(figsize=(9, 7))

for pid, g in plot_df.groupby("pid"):

    # Thin each trajectory so the plot is readable
    g = g.sort_values("absolute_time_hours")

    if len(g) < 3:
        continue

    # Downsample to avoid too many arrows
    step = max(1, len(g) // 25)
    g = g.iloc[::step, :]

    plt.plot(
        g["latent_pc1"],
        g["latent_pc2"],
        alpha=0.35,
        linewidth=1
    )

    plt.scatter(
        g["latent_pc1"],
        g["latent_pc2"],
        s=10,
        alpha=0.5
    )

    # arrows
    for i in range(len(g) - 1):

        x1 = g["latent_pc1"].iloc[i]
        y1 = g["latent_pc2"].iloc[i]
        x2 = g["latent_pc1"].iloc[i + 1]
        y2 = g["latent_pc2"].iloc[i + 1]

        plt.arrow(
            x1,
            y1,
            x2 - x1,
            y2 - y1,
            alpha=0.25,
            length_includes_head=True,
            head_width=0.05,
            linewidth=0.5
        )

plt.xlabel("Latent PC1")
plt.ylabel("Latent PC2")
plt.title("Participant Physiologic State Trajectories")

plt.tight_layout()

out_path = os.path.join(
    output_dir,
    "latent_state_pca_arrows_by_pid.png"
)

plt.savefig(out_path, dpi=300)
plt.close()

print("Saved:", out_path)

loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=["PC1_loading", "PC2_loading", "PC3_loading"]
)

loadings["PC1_abs"] = loadings["PC1_loading"].abs()
loadings["PC2_abs"] = loadings["PC2_loading"].abs()
loadings["PC3_abs"] = loadings["PC3_loading"].abs()

loadings_path = os.path.join(
    output_dir,
    "latent_state_pca_loadings.csv"
)

loadings.to_csv(loadings_path)

print("\nSaved PCA loadings:")
print(loadings_path)

print("\nTop PC1 loadings:")
print(loadings.sort_values("PC1_abs", ascending=False)[["PC1_loading"]].head(15))

print("\nTop PC2 loadings:")
print(loadings.sort_values("PC2_abs", ascending=False)[["PC2_loading"]].head(15))

print("\nDONE")

cluster_cols = [
    "latent_pc1",
    "latent_pc2",
    "latent_pc3"
]

cluster_cols = [c for c in cluster_cols if c in df.columns]

X = df[cluster_cols].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = []

for k in range(2, 8):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=25
    )

    labels = km.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)

    scores.append({
        "k": k,
        "silhouette": sil
    })

scores_df = pd.DataFrame(scores)

print("\nSilhouette scores:")
print(scores_df)

scores_path = os.path.join(
    output_dir,
    "latent_state_cluster_silhouette_scores.csv"
)

scores_df.to_csv(scores_path, index=False)

# Choose best k
best_k = int(
    scores_df.sort_values(
        "silhouette",
        ascending=False
    )["k"].iloc[0]
)

print("\nBest k:", best_k)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)

df["latent_state_cluster"] = kmeans.fit_predict(
    scaler.fit_transform(df[cluster_cols])
)

df.to_csv(cluster_output, index=False)

print("\nSaved clustered dataset:")
print(cluster_output)

print("\nCluster counts:")
print(df["latent_state_cluster"].value_counts().sort_index())

summary_vars = [
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",
    "eda_mean_z",
    "eda_slope_z",
    "temp_mean_z",
    "acc_mag_mean_z",
    "acc_activity_count_z",
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean",
    "env_lessonnumber_mean",
    "env_lessonpct_mean",
    "window_mid_hour"
]

summary_vars = [c for c in summary_vars if c in df.columns]

cluster_summary = (
    df.groupby("latent_state_cluster")[summary_vars]
    .mean()
    .reset_index()
)

summary_path = os.path.join(
    output_dir,
    "latent_state_cluster_summary.csv"
)

cluster_summary.to_csv(summary_path, index=False)

print("\nSaved cluster summary:")
print(summary_path)

print("\nCluster summary:")
print(cluster_summary)

plt.figure(figsize=(8, 6))

sc = plt.scatter(
    df["latent_pc1"],
    df["latent_pc2"],
    c=df["latent_state_cluster"],
    s=10,
    alpha=0.7
)

plt.colorbar(sc, label="Latent state cluster")
plt.xlabel("Latent PC1")
plt.ylabel("Latent PC2")
plt.title("Latent Physiologic State Clusters")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "latent_state_clusters_pca.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:", plot_path)

centroids = (
    df.groupby("latent_state_cluster")[["latent_pc1", "latent_pc2"]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 6))

plt.scatter(
    df["latent_pc1"],
    df["latent_pc2"],
    c=df["latent_state_cluster"],
    s=8,
    alpha=0.25
)

plt.scatter(
    centroids["latent_pc1"],
    centroids["latent_pc2"],
    s=250,
    marker="X",
    edgecolor="black"
)

for _, row in centroids.iterrows():
    plt.text(
        row["latent_pc1"],
        row["latent_pc2"],
        f"State {int(row['latent_state_cluster'])}",
        fontsize=11,
        weight="bold"
    )

plt.xlabel("Latent PC1")
plt.ylabel("Latent PC2")
plt.title("Latent State Centroids")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "latent_state_cluster_centroids.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:", plot_path)

if "subject" in df.columns:

    occupancy = (
        pd.crosstab(
            df["subject"],
            df["latent_state_cluster"],
            normalize="index"
        )
        * 100
    )

    occupancy_path = os.path.join(
        output_dir,
        "latent_state_occupancy_by_subject.csv"
    )

    occupancy.to_csv(occupancy_path)

    print("\nSaved occupancy by subject:")
    print(occupancy_path)

    print("\nState occupancy by subject (%):")
    print(occupancy)

    plt.figure(figsize=(10, 6))

    occupancy.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 6)
    )

    plt.ylabel("Percent of windows")
    plt.xlabel("Subject")
    plt.title("Latent State Occupancy by Subject")
    plt.legend(title="Latent state", fontsize=8)

    plt.tight_layout()

    plot_path = os.path.join(
        output_dir,
        "latent_state_occupancy_by_subject.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.close()

    print("Saved:", plot_path)

if "env_lessonpct_mean" in df.columns:

    df["lesson_progress_bin"] = pd.cut(
        df["env_lessonpct_mean"],
        bins=5
    )

    lesson_occ = (
        pd.crosstab(
            df["lesson_progress_bin"],
            df["latent_state_cluster"],
            normalize="index"
        )
        * 100
    )

    lesson_occ_path = os.path.join(
        output_dir,
        "latent_state_occupancy_by_lesson_progress.csv"
    )

    lesson_occ.to_csv(lesson_occ_path)

    print("\nSaved occupancy by lesson progress:")
    print(lesson_occ_path)

    plt.figure(figsize=(10, 6))

    lesson_occ.plot(
        kind="bar",
        stacked=True,
        figsize=(10, 6)
    )

    plt.ylabel("Percent of windows")
    plt.xlabel("Lesson progress bin")
    plt.title("Latent State Occupancy Across Lesson Progress")
    plt.legend(title="Latent state", fontsize=8)

    plt.tight_layout()

    plot_path = os.path.join(
        output_dir,
        "latent_state_occupancy_by_lesson_progress.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.close()

    print("Saved:", plot_path)

print("\nDONE")

state_labels = {
    0: "Low Environmental Exposure State",
    1: "Standard Classroom State",
    2: "High Activation State",
    3: "Warm/Late-Day State"
}

df["state_label"] = df["latent_state_cluster"].map(state_labels)

sort_cols = ["pid", "session", "window_mid_sec"]
df = df.sort_values(sort_cols)

transition_counts = np.zeros((4, 4))

for _, subdf in df.groupby(["pid", "session"]):

    states = subdf["latent_state_cluster"].dropna().astype(int).values

    for i in range(len(states) - 1):
        transition_counts[states[i], states[i + 1]] += 1

transition_probs = transition_counts / transition_counts.sum(axis=1, keepdims=True)
transition_probs = np.nan_to_num(transition_probs)

transition_df = pd.DataFrame(
    transition_probs,
    index=[state_labels[i] for i in range(4)],
    columns=[state_labels[i] for i in range(4)]
)

transition_path = os.path.join(
    output_dir,
    "latent_state_transition_matrix_corrected.csv"
)

transition_df.to_csv(transition_path)

print("Transition probabilities:")
print(transition_df)

print("\nSaved:")
print(transition_path)

plt.figure(figsize=(10, 8))

sns.heatmap(
    transition_df,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=0,
    vmax=1
)

plt.title("Latent Physiologic State Transition Probabilities")
plt.xlabel("Next State")
plt.ylabel("Current State")

plt.tight_layout()

plot_path = ***
plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:")
print(plot_path)

transition_counts_df = pd.DataFrame(
    transition_counts,
    index=[state_labels[i] for i in range(4)],
    columns=[state_labels[i] for i in range(4)]
)

counts_path = os.path.join(
    output_dir,
    "latent_state_transition_counts_corrected.csv"
)

transition_counts_df.to_csv(counts_path)

print("\nTransition counts:")
print(transition_counts_df)

print("\nSaved:")
print(counts_path)

features = [
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",
    "eda_mean_z",
    "eda_slope_z",
    "temp_mean_z",
    "acc_mag_mean_z",
    "acc_activity_count_z",
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean"
]

results = []

for feature in features:

    groups = []

    for state in sorted(df["latent_state_cluster"].unique()):

        vals = (
            df.loc[
                df["latent_state_cluster"] == state,
                feature
            ]
            .dropna()
        )

        groups.append(vals)

    H, p = kruskal(*groups)

    results.append({
        "feature": feature,
        "H_statistic": H,
        "p": p
    })

results_df = pd.DataFrame(results)

results_df["fdr"] = multipletests(
    results_df["p"],
    method="fdr_bh"
)[1]


print("\nSaved:")
print(output_path)

features = [
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",
    "eda_mean_z",
    "temp_mean_z",
    "acc_mag_mean_z",
    "env_indoorco2_mean",
    "env_indoornoise_mean"
]

state_labels = {
    0: "Low Exposure",
    1: "Standard",
    2: "High Activation",
    3: "Warm/Late-Day"
}

summary = (
    df.groupby("latent_state_cluster")[features]
    .mean()
)

categories = features
N = len(categories)

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw=dict(polar=True)
)

for state in summary.index:

    values = summary.loc[state].tolist()
    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=2,
        label=state_labels[state]
    )

    ax.fill(
        angles,
        values,
        alpha=0.1
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)

plt.title("Physiologic Fingerprints of Latent States")

plt.legend(loc="upper right")

plt.tight_layout()

save_path = os.path.join(
    output_dir,
    "latent_state_radar_plot.png"
)

plt.savefig(save_path, dpi=300)
plt.close()

print("Saved:")
print(save_path)

transition_path = os.path.join(
    output_dir,
    "latent_state_transition_counts_corrected.csv"
)

transitions = pd.read_csv(
    transition_path,
    index_col=0
)

labels = list(transitions.index)

source = []
target = []
value = []

for i, row_label in enumerate(labels):
    for j, col_label in enumerate(labels):

        v = transitions.iloc[i, j]

        if v > 0:

            source.append(i)
            target.append(j)
            value.append(v)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        label=labels
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
)])

fig.update_layout(
    title_text="Latent Physiologic State Transitions",
    font_size=12
)

save_path = os.path.join(
    output_dir,
    "latent_state_sankey.png"
)

fig.write_image(
    save_path,
    width=1600,
    height=900,
    scale=2
)

print("Saved:")
print(save_path)

state_labels = {
    0: "Low Exposure",
    1: "Standard",
    2: "High Activation",
    3: "Warm/Late-Day"
}

df["state_label"] = df["latent_state_cluster"].map(state_labels)

features = [
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",
    "eda_mean_z",
    "temp_mean_z",
    "acc_mag_mean_z",
    "acc_activity_count_z",
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean"
]

features = [c for c in features if c in df.columns]

plot_df = df[["latent_state_cluster", "state_label"] + features].copy()

for col in features:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")
    plot_df[col + "_radar_z"] = (
        plot_df[col] - plot_df[col].mean()
    ) / plot_df[col].std()

radar_features = [c + "_radar_z" for c in features]
summary = (
    plot_df
    .groupby(["latent_state_cluster", "state_label"])[radar_features]
    .mean()
    .reset_index()
)

summary.to_csv(summary_path, index=False)

print("Saved summary:")
print(summary_path)

label_map = {
    "hr_mean_z_radar_z": "HR",
    "ibi_mean_z_radar_z": "IBI",
    "ibi_rmssd_z_radar_z": "RMSSD",
    "eda_mean_z_radar_z": "EDA",
    "temp_mean_z_radar_z": "Skin Temp",
    "acc_mag_mean_z_radar_z": "Activity Mag",
    "acc_activity_count_z_radar_z": "Activity Count",
    "env_indoorco2_mean_radar_z": "CO2",
    "env_indoortemperature_mean_radar_z": "Room Temp",
    "env_indoorhumidity_mean_radar_z": "Humidity",
    "env_indoornoise_mean_radar_z": "Noise"
}

radar_labels = [label_map.get(c, c) for c in radar_features]

N = len(radar_features)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(10, 10))
ax = plt.subplot(111, polar=True)

for _, row in summary.iterrows():

    values = row[radar_features].tolist()
    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=2,
        label=row["state_label"]
    )

    ax.fill(
        angles,
        values,
        alpha=0.08
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)

ax.axhline(0, linewidth=1, linestyle="--")

ax.set_title(
    "Z-Scored Physiologic and Environmental Fingerprints of Latent States",
    pad=25,
    fontsize=14
)

ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.25, 1.10)
)

plt.tight_layout()

plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close()

print("Saved radar plot:")
print(save_path)

print("\nZ-scored state summary:")
print(summary)

target_col = "latent_state_cluster"

y = df[target_col].astype(int)

candidate_features = [
    # physiology
    "hr_mean_z",
    "ibi_mean_z",
    "ibi_rmssd_z",
    "eda_mean_z",
    "eda_slope_z",
    "temp_mean_z",
    "acc_mag_mean_z",
    "acc_activity_count_z",

    # environment
    "env_indoorco2_mean",
    "env_indoortemperature_mean",
    "env_indoorhumidity_mean",
    "env_indoornoise_mean",

    # classroom/time context
    "env_lessonnumber_mean",
    "env_lessonpct_mean",
    "window_mid_hour",
    "week",
    "weekday",
    "subject",
    "room"
]

candidate_features = [c for c in candidate_features if c in df.columns]

X = df[candidate_features].copy()

groups = df["pid"]

print("Feature matrix:", X.shape)
print("Target counts:")
print(y.value_counts().sort_index())

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

models = {
    "MultinomialLogisticRegression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        multi_class="multinomial"
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
}

cv = GroupKFold(n_splits=5)

results = []
all_preds = []

for model_name, model in models.items():

    print("\n==============================")
    print(model_name)
    print("==============================")

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    oof_pred = cross_val_predict(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        method="predict",
        n_jobs=-1
    )

    acc = accuracy_score(y, oof_pred)
    bal_acc = balanced_accuracy_score(y, oof_pred)
    macro_f1 = f1_score(y, oof_pred, average="macro")
    weighted_f1 = f1_score(y, oof_pred, average="weighted")

    print("Accuracy:", round(acc, 4))
    print("Balanced accuracy:", round(bal_acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted F1:", round(weighted_f1, 4))
    print("\nClassification report:")
    print(classification_report(y, oof_pred))

    cm = confusion_matrix(y, oof_pred)

    results.append({
        "model": model_name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

    tmp = df[["pid", "session", "week", "weekday", "subject", "room"]].copy()
    tmp["model"] = model_name
    tmp["true_state"] = y.values
    tmp["pred_state"] = oof_pred
    all_preds.append(tmp)

    # Confusion matrix plot
    plt.figure(figsize=(7, 6))

    plt.imshow(cm, aspect="auto")
    plt.colorbar(label="Count")

    plt.xticks(range(cm.shape[1]), range(cm.shape[1]))
    plt.yticks(range(cm.shape[0]), range(cm.shape[0]))

    plt.xlabel("Predicted state")
    plt.ylabel("True state")
    plt.title(f"{model_name} Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()

    cm_path = os.path.join(
        output_dir,
        f"latent_state_prediction_confusion_matrix_{model_name}.png"
    )

    plt.savefig(cm_path, dpi=300)
    plt.close()

    print("Saved confusion matrix:", cm_path)

results_df = pd.DataFrame(results)
results_df.to_csv(results_path, index=False)

preds_df = pd.concat(all_preds, ignore_index=True)
preds_df.to_csv(preds_path, index=False)

print("\nDONE")
print(results_df)

print("\nSaved:")
print(results_path)
print(preds_path)

df = pd.read_csv(input_path)

state_labels = {
    0: "Low Exposure",
    1: "Standard",
    2: "High Activation",
    3: "Warm/Late-Day"
}

df["state_label"] = df["latent_state_cluster"].map(state_labels)

rows = []

for pid, g in df.groupby("pid"):

    g = g.sort_values(["session", "window_mid_sec"])

    state_counts = g["latent_state_cluster"].value_counts().sort_index()
    state_probs = state_counts / state_counts.sum()

    state_entropy = entropy(state_probs)

    max_entropy = np.log(len(state_labels))
    normalized_entropy = state_entropy / max_entropy

    states = g["latent_state_cluster"].values

    switches = np.sum(states[1:] != states[:-1])
    switch_rate = switches / max(1, len(states) - 1)

    dominant_state = state_counts.idxmax()
    dominant_state_pct = state_counts.max() / state_counts.sum() * 100

    rows.append({
        "pid": pid,
        "n_windows": len(g),
        "state_entropy": state_entropy,
        "normalized_state_entropy": normalized_entropy,
        "switch_count": switches,
        "switch_rate": switch_rate,
        "dominant_state": dominant_state,
        "dominant_state_label": state_labels[dominant_state],
        "dominant_state_percent": dominant_state_pct
    })

entropy_df = pd.DataFrame(rows)

entropy_df.to_csv(entropy_path, index=False)

print("Saved:")
print(entropy_path)

print("\nParticipant entropy summary:")
print(entropy_df.describe(include="all"))

plt.figure(figsize=(10, 5))

plt.bar(
    entropy_df["pid"].astype(str),
    entropy_df["normalized_state_entropy"]
)

plt.xlabel("Participant")
plt.ylabel("Normalized state entropy")
plt.title("Participant Physiologic State Entropy")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "participant_state_entropy_barplot.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:")
print(plot_path)

plt.figure(figsize=(10, 5))

plt.bar(
    entropy_df["pid"].astype(str),
    entropy_df["switch_rate"]
)

plt.xlabel("Participant")
plt.ylabel("Switch rate")
plt.title("Participant Latent State Switching Rate")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "participant_state_switch_rate_barplot.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:")
print(plot_path)

plt.figure(figsize=(7, 6))

plt.scatter(
    entropy_df["normalized_state_entropy"],
    entropy_df["switch_rate"],
    s=80
)

for _, row in entropy_df.iterrows():
    plt.text(
        row["normalized_state_entropy"],
        row["switch_rate"],
        str(int(row["pid"])),
        fontsize=8
    )

plt.xlabel("Normalized state entropy")
plt.ylabel("Switch rate")
plt.title("Physiologic Flexibility: Entropy vs Switching")

plt.tight_layout()

plot_path = os.path.join(
    output_dir,
    "participant_entropy_vs_switching.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("Saved:")
print(plot_path)

print("\nDONE")

entropy_df = pd.read_csv(entropy_path)
state_df = pd.read_csv(state_path)

print("Entropy shape:", entropy_df.shape)
print("State shape:", state_df.shape)

entropy_df["dominant_state_label"] = (
    entropy_df["dominant_state_label"]
    .astype(str)
    .str.strip()
)

summary = (
    entropy_df
    .groupby("dominant_state_label")
    .agg({
        "normalized_state_entropy": ["mean", "std", "median"],
        "switch_rate": ["mean", "std", "median"],
        "dominant_state_percent": ["mean", "std"]
    })
)

summary.columns = [
    "_".join(col).strip()
    for col in summary.columns.values
]

summary = summary.reset_index()

print("\nSUMMARY BY DOMINANT STATE")
print(summary)

summary.to_csv(
    os.path.join(
        output_dir,
        "participant_flexibility_by_dominant_state.csv"
    ),
    index=False
)

results = []

for variable in [
    "normalized_state_entropy",
    "switch_rate",
    "dominant_state_percent"
]:

    groups = []

    for state in entropy_df["dominant_state_label"].unique():

        vals = entropy_df.loc[
            entropy_df["dominant_state_label"] == state,
            variable
        ].dropna()

        if len(vals) > 0:
            groups.append(vals)

    H, p = kruskal(*groups)

    results.append({
        "variable": variable,
        "H_statistic": H,
        "p": p
    })

results_df = pd.DataFrame(results)

results_df["fdr"] = multipletests(
    results_df["p"],
    method="fdr_bh"
)[1]

print("\nKRUSKAL RESULTS")
print(results_df)

results_df.to_csv(
    os.path.join(
        output_dir,
        "participant_flexibility_statistics.csv"
    ),
    index=False
)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=entropy_df,
    x="dominant_state_label",
    y="normalized_state_entropy"
)

plt.title(
    "Participant State Entropy by Dominant State",
    fontsize=14,
    weight="bold"
)

plt.xlabel("Dominant State")
plt.ylabel("Normalized State Entropy")

plt.xticks(rotation=15)

plt.tight_layout()

entropy_plot_path = os.path.join(
    output_dir,
    "participant_entropy_by_dominant_state.png"
)

plt.savefig(
    entropy_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("\nSaved:", entropy_plot_path)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=entropy_df,
    x="dominant_state_label",
    y="switch_rate"
)

plt.title(
    "Participant Switching Rate by Dominant State",
    fontsize=14,
    weight="bold"
)

plt.xlabel("Dominant State")
plt.ylabel("Switch Rate")

plt.xticks(rotation=15)

plt.tight_layout()

switch_plot_path = os.path.join(
    output_dir,
    "participant_switch_rate_by_dominant_state.png"
)

plt.savefig(
    switch_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved:", switch_plot_path)

subject_entropy = (
    state_df
    .groupby(["pid", "subject"])["latent_state_cluster"]
    .nunique()
    .reset_index(name="n_unique_states")
)

subject_summary = (
    subject_entropy
    .groupby("subject")["n_unique_states"]
    .agg(["mean", "std", "median", "count"])
    .reset_index()
)

print("\nSUBJECT FLEXIBILITY")
print(subject_summary)

subject_summary.to_csv(
    os.path.join(
        output_dir,
        "subject_level_state_flexibility.csv"
    ),
    index=False
)

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=subject_entropy,
    x="subject",
    y="n_unique_states"
)

plt.title(
    "State Diversity Across Subjects",
    fontsize=14,
    weight="bold"
)

plt.xlabel("Subject")
plt.ylabel("Number of Unique States")

plt.xticks(rotation=30)

plt.tight_layout()

subject_plot_path = os.path.join(
    output_dir,
    "subject_state_diversity_boxplot.png"
)

plt.savefig(
    subject_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved:", subject_plot_path)

entropy_df["high_flexibility"] = (
    entropy_df["normalized_state_entropy"]
    >= entropy_df["normalized_state_entropy"].median()
).astype(int)

high_flex_summary = (
    entropy_df
    .groupby("dominant_state_label")["high_flexibility"]
    .mean()
    .reset_index()
)

high_flex_summary = high_flex_summary.rename(
    columns={"high_flexibility": "percent_high_flexibility"}
)

high_flex_summary["percent_high_flexibility"] *= 100

print("\nHIGH FLEXIBILITY %")
print(high_flex_summary)

high_flex_summary.to_csv(
    os.path.join(
        output_dir,
        "high_flexibility_by_state.csv"
    ),
    index=False
)

print("\nDONE")

lesson_col = "env_lessonpct_mean"

variables = {
    "hr_mean_z": "Heart Rate",
    "ibi_rmssd_z": "RMSSD",
    "eda_mean_z": "EDA",
    "acc_mag_mean_z": "Activity Magnitude"
}

plot_df = df.copy()
plot_df = plot_df[plot_df[lesson_col].notna()]
plot_df = plot_df[plot_df[lesson_col] >= 0]
plot_df = plot_df[plot_df[lesson_col] <= 1]

plot_df["lesson_percent"] = plot_df[lesson_col] * 100

n_bins = 20
plot_df["lesson_bin"] = pd.cut(
    plot_df["lesson_percent"],
    bins=np.linspace(0, 100, n_bins + 1),
    include_lowest=True
)

plot_df["lesson_bin_mid"] = plot_df["lesson_bin"].apply(lambda x: x.mid).astype(float)

summary_rows = []

for var, label in variables.items():
    if var not in plot_df.columns:
        print(f"WARNING: {var} not found. Skipping.")
        continue

    temp = (
        plot_df
        .groupby("lesson_bin_mid")[var]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    temp["sem"] = temp["std"] / np.sqrt(temp["count"])
    temp["variable"] = var
    temp["label"] = label
    summary_rows.append(temp)

summary = pd.concat(summary_rows, ignore_index=True)

summary.to_csv(
    os.path.join(output_dir, "lesson_progress_trajectory_summary.csv"),
    index=False
)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True)
axes = axes.flatten()

panel_labels = ["A", "B", "C", "D"]

for i, (ax, (var, label)) in enumerate(zip(axes, variables.items())):
    temp = summary[summary["variable"] == var].copy()

    temp["mean_smooth"] = (
        temp["mean"]
        .rolling(window=2, center=True, min_periods=1)
        .mean()
    )

    temp["sem_smooth"] = (
        temp["sem"]
        .rolling(window=2, center=True, min_periods=1)
        .mean()
    )

    x = temp["lesson_bin_mid"].astype(float).values
    y = temp["mean_smooth"].astype(float).values
    sem = temp["sem_smooth"].astype(float).values

    ax.plot(x, y, marker="o", linewidth=2.7, markersize=5)
    ax.fill_between(x, y - sem, y + sem, alpha=0.12)

    ax.axhline(0, linestyle="-", linewidth=0.8, alpha=0.4)
    ax.axvline(50, linestyle="--", linewidth=1.0, alpha=0.45)

    ax.text(
        0.02,
        0.95,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

    ax.set_title(label, fontsize=14)
    ax.set_ylabel("Z-score", fontsize=12)
    ax.grid(alpha=0.25)

for ax in axes[2:]:
    ax.set_xlabel("Lesson Progress (%)", fontsize=12)

for ax in axes:
    ax.set_xlim(0, 55)
    ax.set_xticks([0, 10, 20, 30, 40, 50])
    ax.tick_params(axis="both", labelsize=10)

fig.suptitle(
    "Physiologic Trajectories Across Lesson Progression",
    fontsize=18,
    y=1.02
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

output_path = os.path.join(
    output_dir,
    "lesson_progress_trajectory_multipanel_clean.png"
)

plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(output_path)
print(os.path.join(output_dir, "lesson_progress_trajectory_summary.csv"))

SyntaxError: invalid syntax (1312991499.py, line 1)